# 🔬 Bloque 2 — Sesión 2: Detección de normalidad y transformaciones

**Módulo:** Depuración, Limpieza y Clasificación de Datos (5109)  
**Unidad Didáctica:** UD2 — Verificación estadística de la calidad de los datos  
**Duración estimada:** ~3 horas

---

### Contenidos de esta sesión
- **2.3** Detección de normalidad: test de Shapiro-Wilk y gráfico Q-Q
- **2.4** Transformaciones para normalizar datos: logarítmica, raíz cuadrada y Box-Cox

### Criterios de evaluación asociados
> **(CE a)** Se han descrito las técnicas estadísticas de análisis de la calidad técnica de los datos.  
> **(CE b)** Se han aplicado técnicas de evaluación de la calidad de los datos basadas en cálculos estadísticos para detectar la cobertura y el sesgo.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from scipy.stats import shapiro, normaltest, probplot, boxcox
from sklearn.preprocessing import PowerTransformer

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 110
np.random.seed(42)

titanic = sns.load_dataset('titanic')
diamonds = sns.load_dataset('diamonds')
penguins = sns.load_dataset('penguins').dropna()

print('✅ Librerías y datasets cargados')

---
## 2.3 Detección de normalidad

En la sesión anterior aprendimos a identificar visualmente el sesgo y la curtosis. Ahora añadimos dos herramientas más rigurosas para comprobar si una variable sigue una distribución normal: el **test de Shapiro-Wilk** y el **gráfico Q-Q**.

### ¿Por qué comprobar la normalidad?

Muchos métodos estadísticos y algunos algoritmos de ML asumen normalidad:
- Tests paramétricos (t-test, ANOVA): asumen normalidad de los grupos
- Regresión lineal: asume residuos normales
- LDA (Análisis Discriminante Lineal): asume normalidad por clase

Si la variable no es normal y el método lo requiere, podemos aplicar una transformación para acercarnos a la normalidad antes de entrenar.

### Test de Shapiro-Wilk

El test de Shapiro-Wilk contrasta:
- **H₀ (hipótesis nula):** los datos provienen de una distribución normal
- **H₁ (hipótesis alternativa):** los datos NO provienen de una distribución normal

El test devuelve un **estadístico W** (entre 0 y 1, cuanto más cerca de 1 más normal) y un **p-valor**:

- Si **p-valor > 0.05** → no rechazamos H₀ → *compatible* con normalidad
- Si **p-valor ≤ 0.05** → rechazamos H₀ → la distribución **no es normal**

> ⚠️ Limitación: con muestras grandes (n > 5.000), el test detecta desviaciones mínimas e irrelevantes como significativas. En esos casos, combinar con el Q-Q plot es imprescindible.

In [ ]:
# Test de Shapiro-Wilk sobre distintas distribuciones
np.random.seed(42)

muestras = {
    'Normal N(0,1)':      np.random.normal(0, 1, 200),
    'Uniforme U(0,1)':    np.random.uniform(0, 1, 200),
    'Log-normal':         np.random.lognormal(0, 1, 200),
    'Exponencial':        np.random.exponential(1, 200),
}

print(f"{'Variable':<20} {'Estadístico W':>15} {'p-valor':>12} {'Normalidad?':>15}")
print('─' * 65)
for nombre, datos in muestras.items():
    W, p = shapiro(datos)
    resultado = '✅ Compatible' if p > 0.05 else '❌ NO normal'
    print(f"{nombre:<20} {W:>15.4f} {p:>12.4f} {resultado:>15}")

In [ ]:
# Shapiro-Wilk sobre variables reales — cuidado con n grande
print("── Titanic (n ≈ 714-891) ──")
for col in ['age', 'fare']:
    serie = titanic[col].dropna()
    W, p = shapiro(serie[:500])  # limitamos a 500 para que el test sea informativo
    resultado = '✅ Compatible' if p > 0.05 else '❌ NO normal'
    print(f"  {col:<10} W={W:.4f}  p={p:.5f}  {resultado}  (n={len(serie[:500])})")

print()
print("── Penguins (n ≈ 333) ──")
for col in ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']:
    serie = penguins[col].dropna()
    W, p = shapiro(serie)
    resultado = '✅ Compatible' if p > 0.05 else '❌ NO normal'
    print(f"  {col:<22} W={W:.4f}  p={p:.5f}  {resultado}")

### Gráfico Q-Q (Quantile-Quantile plot)

El **Q-Q plot** compara los cuantiles empíricos de los datos con los cuantiles teóricos de la distribución normal. Si los datos son normales, los puntos caen sobre la **línea diagonal de referencia**.

- **Puntos en la línea:** distribución normal
- **Puntos en forma de S:** sesgo
- **Colas elevadas/bajas:** curtosis elevada/baja

El Q-Q plot es especialmente útil con muestras grandes donde Shapiro-Wilk pierde sensibilidad práctica.

In [ ]:
# Q-Q plot para distintas distribuciones
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

distribuciones = {
    'Normal N(0,1)':   np.random.normal(0, 1, 500),
    'Log-normal':      np.random.lognormal(0, 0.8, 500),
    'Uniforme U(0,1)': np.random.uniform(0, 1, 500),
    'Bimodal':         np.concatenate([np.random.normal(-2, 0.8, 250),
                                       np.random.normal(2, 0.8, 250)])
}

for ax, (nombre, datos) in zip(axes.flat, distribuciones.items()):
    res = probplot(datos, dist='norm')
    teoricos = res[0][0]
    empiricos = res[0][1]
    slope, intercept, r = res[1]
    
    ax.scatter(teoricos, empiricos, s=10, alpha=0.5, color='steelblue')
    x_line = np.array([teoricos.min(), teoricos.max()])
    ax.plot(x_line, slope * x_line + intercept, 'crimson', linewidth=2, label='Línea normal')
    ax.set_title(f'{nombre}
R² = {r**2:.4f}')
    ax.set_xlabel('Cuantiles teóricos (Normal)')
    ax.set_ylabel('Cuantiles empíricos')
    ax.legend(fontsize=8)

plt.suptitle('Gráficos Q-Q: comparación de distribuciones', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
def panel_normalidad(serie, nombre=''):
    """Combina histograma + KDE y Q-Q plot para diagnóstico completo de normalidad."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    
    # Histograma con curva KDE y normal ajustada
    mu, sigma = serie.mean(), serie.std()
    axes[0].hist(serie, bins=30, color='steelblue', edgecolor='white',
                 alpha=0.7, density=True, label='Datos')
    serie.plot.kde(ax=axes[0], color='navy', linewidth=2, label='KDE')
    x = np.linspace(serie.min(), serie.max(), 300)
    axes[0].plot(x, stats.norm.pdf(x, mu, sigma), 'crimson',
                 linewidth=2, linestyle='--', label=f'Normal(μ={mu:.1f}, σ={sigma:.1f})')
    axes[0].set_title(f'{nombre} — Histograma')
    axes[0].legend(fontsize=8)
    
    # Q-Q plot
    res = probplot(serie, dist='norm')
    teoricos, empiricos = res[0]
    slope, intercept, r = res[1]
    axes[1].scatter(teoricos, empiricos, s=12, alpha=0.5, color='steelblue')
    x_line = np.array([teoricos.min(), teoricos.max()])
    axes[1].plot(x_line, slope * x_line + intercept, 'crimson', linewidth=2)
    
    W, p = shapiro(serie[:500] if len(serie) > 500 else serie)
    skew = serie.skew()
    axes[1].set_title(f'{nombre} — Q-Q plot
Shapiro W={W:.4f} p={p:.4f} | Sesgo={skew:.3f}')
    axes[1].set_xlabel('Cuantiles teóricos')
    axes[1].set_ylabel('Cuantiles empíricos')
    
    plt.tight_layout()
    plt.show()

panel_normalidad(titanic['fare'].dropna(), 'Tarifa Titanic')

---
## 2.4 Transformaciones para normalizar datos

Cuando una variable tiene sesgo importante, aplicar una **transformación matemática** puede acercarla a la normalidad. Esto beneficia a los modelos que asumen normalidad y puede mejorar la interpretabilidad.

### ¿Cuándo transformar?

- |Sesgo| > 1 y el algoritmo asume normalidad (regresión lineal, LDA, t-test)
- La variable tiene colas muy pesadas que generan outliers problemáticos
- Se quiere mejorar la linealidad de la relación con la variable objetivo

> ⚠️ No todos los algoritmos necesitan normalidad. Los basados en árboles (Random Forest, XGBoost) son invariantes a monotransformaciones. Transforma solo cuando tenga sentido para el método elegido.

### Transformación logarítmica

La transformación `log(x)` comprime los valores grandes y expande los pequeños, corrigiendo el sesgo positivo (cola derecha).

**Requisito:** la variable debe ser **estrictamente positiva** (x > 0).  
Se suele usar `log(x + 1)` = `log1p(x)` para incluir ceros.

In [ ]:
fare = titanic['fare'].dropna()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Original
axes[0].hist(fare, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Original  |  Sesgo = {fare.skew():.3f}')
axes[0].set_xlabel('Tarifa (£)')

# Transformada log
fare_log = np.log1p(fare)
axes[1].hist(fare_log, bins=40, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[1].set_title(f'log(fare + 1)  |  Sesgo = {fare_log.skew():.3f}')
axes[1].set_xlabel('log(Tarifa + 1)')

plt.suptitle('Transformación logarítmica — Tarifa Titanic', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

W_orig, p_orig = shapiro(fare[:500])
W_log, p_log = shapiro(fare_log[:500])
print(f"Original  → W={W_orig:.4f}, p={p_orig:.6f}")
print(f"log1p     → W={W_log:.4f}, p={p_log:.6f}")

### Transformación raíz cuadrada

La transformación `√x` también corrige el sesgo positivo, pero de forma más suave que el logaritmo. Útil cuando los valores incluyen ceros y el sesgo no es muy extremo.

**Requisito:** x ≥ 0.

In [ ]:
price = diamonds['price']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Original
axes[0].hist(price, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Original
Sesgo = {price.skew():.3f}')

# Raíz cuadrada
price_sqrt = np.sqrt(price)
axes[1].hist(price_sqrt, bins=40, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_title(f'√price
Sesgo = {price_sqrt.skew():.3f}')

# Log
price_log = np.log1p(price)
axes[2].hist(price_log, bins=40, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[2].set_title(f'log(price+1)
Sesgo = {price_log.skew():.3f}')

for ax in axes:
    ax.set_xlabel('Precio (£)')

plt.suptitle('Comparación de transformaciones — Precio Diamonds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Transformación Box-Cox

La transformación Box-Cox es la más flexible: busca automáticamente el exponente λ que hace la distribución más cercana a la normal:

$$y = \begin{cases} \frac{x^\lambda - 1}{\lambda} & \text{si } \lambda \neq 0 \\ \ln(x) & \text{si } \lambda = 0 \end{cases}$$

- λ = 1 → sin transformación
- λ = 0 → logaritmo natural
- λ = 0.5 → raíz cuadrada
- λ = −1 → inversa (1/x)

**Ventaja:** el parámetro λ se estima con máxima verosimilitud, optimizando la normalidad.  
**Requisito:** x > 0 (estrictamente positivo).

In [ ]:
# Box-Cox sobre el precio de los diamantes (positivo, sin ceros)
price_vals = diamonds['price'].values.astype(float)

price_bc, lambda_optimo = boxcox(price_vals)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].hist(price_vals, bins=40, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title(f'Original
Sesgo = {stats.skew(price_vals):.3f}')
axes[0].set_xlabel('Precio')

axes[1].hist(np.log1p(price_vals), bins=40, color='coral', edgecolor='white', alpha=0.8)
axes[1].set_title(f'log(price+1)
Sesgo = {stats.skew(np.log1p(price_vals)):.3f}')
axes[1].set_xlabel('log(Precio+1)')

axes[2].hist(price_bc, bins=40, color='mediumseagreen', edgecolor='white', alpha=0.8)
axes[2].set_title(f'Box-Cox (λ={lambda_optimo:.3f})
Sesgo = {stats.skew(price_bc):.3f}')
axes[2].set_xlabel('Box-Cox(Precio)')

plt.suptitle('Box-Cox vs. logaritmo — Precio Diamonds', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"λ óptimo encontrado: {lambda_optimo:.4f}")
print(f"Interpretación: λ ≈ {lambda_optimo:.2f} → transformación entre log y raíz cuadrada")

In [ ]:
# PowerTransformer de scikit-learn: Box-Cox y Yeo-Johnson (acepta negativos)
# Yeo-Johnson es la alternativa cuando hay valores ≤ 0
from sklearn.preprocessing import PowerTransformer

fare = titanic['fare'].dropna().values.reshape(-1, 1)

pt_bc = PowerTransformer(method='box-cox')       # requiere x > 0
pt_yj = PowerTransformer(method='yeo-johnson')   # acepta cualquier valor

fare_bc = pt_bc.fit_transform(fare).ravel()
fare_yj = pt_yj.fit_transform(fare).ravel()

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, datos, titulo, color in zip(
    axes,
    [fare.ravel(), fare_bc, fare_yj],
    [f'Original
Sesgo={stats.skew(fare.ravel()):.3f}',
     f'Box-Cox (λ={pt_bc.lambdas_[0]:.3f})
Sesgo={stats.skew(fare_bc):.3f}',
     f'Yeo-Johnson (λ={pt_yj.lambdas_[0]:.3f})
Sesgo={stats.skew(fare_yj):.3f}'],
    ['steelblue', 'coral', 'mediumseagreen']
):
    ax.hist(datos, bins=35, color=color, edgecolor='white', alpha=0.8)
    ax.set_title(titulo)

plt.suptitle('PowerTransformer (sklearn) — Tarifa Titanic', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Función integral: diagnóstico + transformación recomendada
def recomendar_transformacion(serie, nombre=''):
    """Evalúa la normalidad y recomienda la transformación más adecuada."""
    sesgo = serie.skew()
    tiene_ceros = (serie == 0).any()
    tiene_negativos = (serie < 0).any()
    
    print(f"Variable: {nombre or serie.name}")
    print(f"  Sesgo original: {sesgo:.3f}")
    print(f"  Valores cero: {tiene_ceros}  |  Valores negativos: {tiene_negativos}")
    
    if abs(sesgo) < 0.5:
        print("  → Distribución suficientemente simétrica. No es necesario transformar.")
    elif tiene_negativos:
        print("  → Recomendación: Yeo-Johnson (acepta negativos)")
        pt = PowerTransformer(method='yeo-johnson')
        transformada = pt.fit_transform(serie.values.reshape(-1, 1)).ravel()
        print(f"  → Sesgo tras Yeo-Johnson: {stats.skew(transformada):.3f}  (λ={pt.lambdas_[0]:.3f})")
    elif tiene_ceros:
        print("  → Recomendación: log(x+1) o Yeo-Johnson (hay ceros, Box-Cox no aplica)")
        transformada = np.log1p(serie)
        print(f"  → Sesgo tras log1p: {stats.skew(transformada):.3f}")
    else:
        print("  → Recomendación: Box-Cox (todos los valores son positivos)")
        transformada, lam = boxcox(serie.values)
        print(f"  → Sesgo tras Box-Cox: {stats.skew(transformada):.3f}  (λ={lam:.3f})")
    print()

for col in ['fare', 'age']:
    recomendar_transformacion(titanic[col].dropna(), col)
for col in ['price', 'carat']:
    recomendar_transformacion(diamonds[col], col)

---
## 🔧 Ejercicios

### Ejercicio 1 — Diagnóstico de normalidad completo

Aplica la función `panel_normalidad()` a las variables `bill_length_mm` y `body_mass_g` del dataset `penguins`. Interpreta los Q-Q plots: ¿los puntos siguen la línea? ¿Hay colas divergentes?

In [ ]:
# panel_normalidad(penguins['bill_length_mm'].dropna(), 'Longitud pico (Penguins)')
# panel_normalidad(penguins['body_mass_g'].dropna(), 'Masa corporal (Penguins)')

# ¿Qué revela el Q-Q plot de bill_length_mm?
# Pista: recuerda que hay 3 especies. ¿Podría ser bimodal?

### Ejercicio 2 — Elegir la mejor transformación

Para la variable `carat` del dataset `diamonds`, prueba las tres transformaciones (log1p, sqrt y Box-Cox). Para cada una:
1. Calcula el sesgo resultante
2. Aplica el test de Shapiro-Wilk (usa solo los primeros 500 valores)
3. Dibuja el Q-Q plot

¿Qué transformación produce el resultado más cercano a la normalidad?

In [ ]:
# carat = diamonds['carat'].values
# Prueba: log1p, sqrt y Box-Cox
# Compara sesgo + Shapiro + Q-Q en un panel de 3 columnas
# ...


### Ejercicio 3 — Pipeline de normalización

Usando `recomendar_transformacion()`, analiza las variables numéricas del dataset `penguins`. Para las que necesiten transformación, aplícala y verifica que el sesgo mejora. Documenta tus decisiones en comentarios: ¿usarías estas variables transformadas en un modelo de regresión?

In [ ]:
# cols_peng = ['bill_length_mm', 'bill_depth_mm', 'flipper_length_mm', 'body_mass_g']
# for col in cols_peng:
#     recomendar_transformacion(penguins[col].dropna(), col)

# Aplica las transformaciones recomendadas y muestra el sesgo antes/después en una tabla
# ...


---
## 📝 Resumen de la sesión

| Concepto | Clave |
|----------|-------|
| **Shapiro-Wilk** | Test de normalidad. p > 0.05 → compatible con normal. Pierde potencia con n > 5.000. |
| **Q-Q plot** | Visual: puntos sobre la diagonal = normal. Colas divergentes = colas pesadas / sesgo. |
| **log(x+1)** | Corrige sesgo positivo fuerte. Requiere x ≥ 0. Simple e interpretable. |
| **√x** | Corrige sesgo positivo moderado. Requiere x ≥ 0. Efecto más suave que log. |
| **Box-Cox** | Encuentra λ óptimo automáticamente. Requiere x > 0. Implementado en scipy y sklearn. |
| **Yeo-Johnson** | Como Box-Cox pero acepta negativos y ceros. Disponible en `PowerTransformer`. |
| **¿Cuándo transformar?** | \|Sesgo\| > 1 + el modelo asume normalidad. No siempre necesario (árboles, etc.). |

### 🔜 Próxima sesión
**Bloque 3 — Sesión 1:** Correlación entre variables. Pearson, Spearman y Cramér's V. P-valor y significación estadística.